In [1]:
import numpy as np
import pandas as pd
from datetime import datetime as dt
from datetime import timedelta as td
import random

In [2]:
# Define random state for NumPy
np.random.seed(42)

# Define random state for Python
random.seed(42)

In [3]:
# Create the columns of the patients_df dataframe

# Create continuous patient IDs for 1000 patients
patient_ids = np.arange(start=1, stop=1001)

# Create whole number (integer) patient age following normal dist with mean=55 and sd=12
patient_ages = np.random.normal(loc=55, scale=12, size=1000).astype(int)

# Create patient sex list
patient_sexes = np.random.choice(['M', 'F', 'Other'], size=1000, p=[0.50, 0.48, 0.02])

# Create patient condition list
patient_conditions = np.random.choice(['Lower Back Pain', 'Knee Osteoarthritis', 'Post-Op Shoulder', 'Neck Pain'], size=1000, p=[0.4, 0.3, 0.2, 0.1])

In [4]:
# Create patients dataframe (patients_df)
patients_df = pd.DataFrame({
    'patient_id': patient_ids,
    'age': patient_ages,
    'gender': patient_sexes,
    'condition': patient_conditions
})

In [5]:
patients_df.head()

,patient_id,age,gender,condition
0,1,60,M,Lower Back Pain
1,2,53,M,Lower Back Pain
2,3,62,F,Lower Back Pain
3,4,73,F,Lower Back Pain
4,5,52,M,Post-Op Shoulder


In [6]:
# Create the columns of the plans_df dataframe

# Create continuous plan IDs for 1000 patients
plan_ids = np.arange(1000, 2000)

# Create treatment start dates throughout 2025
start_dates = [dt(2025, 1, 1) + td(days=random.randint(0, 364)) for i in range(1000)]

# Create treatment ending dates (56 days or 8 weeks after start date)
end_dates = [start_date + td(days=56) for start_date in start_dates]

In [7]:
# Create treatment plans dataframe (plans_df)
plans_df = pd.DataFrame({
    'plan_id': plan_ids,
    'patient_id': patient_ids,
    'start_date': start_dates,
    'end_date': end_dates,
    'prescribed_sessions_per_week': 3
})

In [8]:
plans_df.head()

,plan_id,patient_id,start_date,end_date,prescribed_sessions_per_week
0,1000,1,2025-11-24,2026-01-19,3
1,1001,2,2025-02-27,2025-04-24,3
2,1002,3,2025-01-13,2025-03-10,3
3,1003,4,2025-05-21,2025-07-16,3
4,1004,5,2025-05-06,2025-07-01,3


In [9]:
# Create empty list of lists (rows) to build the sessions and assessments dataframes
sessions_list = []
assessments_list = []
session_id_counter = 50000
assessment_id_counter = 1000

In [10]:
# Iterate throught the rows of the plans_df to return row index (i) and row data (row)
for i, row in plans_df.iterrows():

    # Store that row column value in a variable
    pid = row['patient_id']
    plid = row['plan_id']
    sdate = row['start_date']
    
    # Generate bimodal patient adherence probability. High adherrence represents 70% of patients and low adherence represents 30% of patients.
    adherence_prob = np.random.beta(a=5, b=2) if np.random.random() > 0.3 else np.random.beta(a=2, b=5)

    # Generate patient baseline pain assessment
    baseline_pain = np.random.randint(6, 10)
    assessments_list.append([assessment_id_counter, pid, sdate, 'Baseline', baseline_pain])
    assessment_id_counter = assessment_id_counter + 1
    current_pain = baseline_pain

    # Generate 8 weeks of session data (3 possible sessions per week = 24 total sessions)
    for week in range(8):
        for session_num in range(3):
            session_date = sdate + td(days=(week*7 + session_num*2))
            
            # Did the patient do the session?
            completed = 1 if np.random.random() < adherence_prob else 0
            
            if completed==1:
                duration = int(np.random.normal(loc=20, scale=3))
                pain_pre = current_pain
                # Add chance of random pain drop after session
                pain_post = max(pain_pre - np.random.randint(0, 3), 0)
                sessions_list.append([session_id_counter, plid, session_date, duration, completed, pain_pre, pain_post])
            else:
                sessions_list.append([session_id_counter, plid, session_date, 0, completed, np.nan, np.nan])
                
            session_id_counter = session_id_counter + 1
            
        # End of week: simulate recovery. High adherence = faster pain drop.
        if adherence_prob > 0.7:
            current_pain = max(current_pain - random.uniform(0.5, 1.2), 0)
        else:
            current_pain = max(current_pain - random.uniform(0, 0.4), 0)

    # Generate patient final pain assessment
    final_date = sdate + td(days=56)
    final_pain = round(current_pain)
    assessments_list.append([assessment_id_counter, pid, final_date, 'Final', final_pain])
    assessment_id_counter = assessment_id_counter + 1

In [11]:
# Create sessions dataframe (sessions_df)
sessions_df = pd.DataFrame(data=sessions_list, columns=['session_id', 'plan_id', 'session_date', 'duration_minutes', 'completed', 'pain_pre', 'pain_post'])

In [12]:
sessions_df.head()

,session_id,plan_id,session_date,duration_minutes,completed,pain_pre,pain_post
0,50000,1000,2025-11-24,0,0,NaN,NaN
1,50001,1000,2025-11-26,17,1,9.000000,8.000000
2,50002,1000,2025-11-28,19,1,9.000000,7.000000
3,50003,1000,2025-12-01,15,1,8.118806,6.118806
4,50004,1000,2025-12-03,15,1,8.118806,6.118806


In [13]:
# Create assessments dataframe (assessments_df)
assessments_df = pd.DataFrame(data=assessments_list, columns=['assessment_id', 'patient_id', 'assessment_date', 'assessment_type', 'pain_score'])

In [14]:
assessments_df.head()

,assessment_id,patient_id,assessment_date,assessment_type,pain_score
0,1000,1,2025-11-24,Baseline,9
1,1001,1,2026-01-19,Final,2
2,1002,2,2025-02-27,Baseline,8
3,1003,2,2025-04-24,Final,1
4,1004,3,2025-01-13,Baseline,6


In [15]:
# Export dataframes to CSV file
patients_df.to_csv('patients.csv', index=False)
plans_df.to_csv('treatment_plans.csv', index=False)
sessions_df.to_csv('session_logs.csv', index=False)
assessments_df.to_csv('clinical_assessments.csv', index=False)